## CASH Notebook

## Celestial Chase - LVL 2: Ghost Stars

### 🛰️ Need help? Open the mission briefing:
[**OPEN LVL 2 HINT PAGE**](https://alexrtw05.github.io/CASH-project/lvl2.html)

_Open in your browser for BGR explainers, cluster tools, and the key calculator._

The Astrophage trail leads deeper into the system. Four planets. Four signals. None of them quite white.

Your instruments detect four near-white pixels scattered across the nebula. They look like noise - but they are not. Each one was placed by a planet at the exact moment of observation. Their blue channel encodes the distance. Their position in the sky encodes the altitude.

Combine both to find the key.

---

**The encoded signal:** `lpoo`

**Your task:**
1. Display the nebula and find the **four planet marker pixels** using `cv2`
   - Filter: `G == 255` and `R == 255` and `B != 255` (pure white decoys have B=255, real markers don't)
   - Each planet leaves a 3x3 marker - take the center of each cluster
2. For each center pixel, read its **blue channel**: `img[y, x, 0]` (OpenCV uses BGR)
3. Use `ephem` to compute each planet's **altitude in degrees** on `2025/6/21 12:00:00` UTC from Zurich
   Planets in order: `['Mars', 'Jupiter', 'Saturn', 'Mercury']`
4. Match each cluster to its planet using the position formula:
   ```
   x = int((az_deg % 360) / 360 * 600)
   y = int((90 - alt_deg) / 180 * 600)
   ```
5. Build the key: `key = sum(blue_channel[i] + int(alt_deg[i]) for each planet)`
6. Build the permutation and reverse the transposition


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

img_b64 = "..."

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('nebula.png', img)
display(IPImage('nebula.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random, math, numpy as np

encoded      = "lpoo"
obs_date     = "2025/6/21 12:00:00"
obs_lat      = "47.3769"
obs_lon      = "8.5417"
planet_names = ['Mars', 'Jupiter', 'Saturn', 'Mercury']
W, H         = 600, 600

# TODO Step 1: Find all near-white pixels (G==255 and R==255) in img
# Group into 4 clusters and take the center of each
# Hint: np.where((img[:,:,1]==255) & (img[:,:,2]==255))

# TODO Step 2: For each cluster center (px, py) read blue channel
# blue = img[py, px, 0]

# TODO Step 3: Compute each planet's az and alt with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date

body_map = {
    'Mars':    ephem.Mars,
    'Jupiter': ephem.Jupiter,
    'Saturn':  ephem.Saturn,
    'Mercury': ephem.Mercury,
}

# Match each planet to its cluster using:
# x = int((az_deg % 360) / 360 * W)
# y = int((90 - alt_deg) / 180 * H)

# TODO Step 4: Build the key
# key = sum(blue_channel[i] + int(alt_deg[i]) for each planet in planet_names order)
key = 0  # replace

# TODO Step 5: Decode
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

def transpose_decode(encoded, perm):
    pass  # decoded[i] = encoded[perm[i]]

answer = transpose_decode(encoded, perm)
print(answer)
